# Local GraphRAG vs Plain Vector RAG: Empirical Evaluation
### Benchmarking Multi-Hop Reasoning & Hallucination Rates on Consumer Hardware (6GB VRAM)
**Author:** Harsh Maurya (harshverse-svg)  
**Institution:** Department of Computer Science & Engineering, IIMT University  
**Environment:** Ubuntu 22.04 LTS on WSL2 / Windows 11, NVIDIA RTX 3060 Laptop (6GB VRAM), Ollama v0.3+

---

## 1. Abstract & Motivation
Standard dense-vector Retrieval-Augmented Generation (RAG) relies on cosine similarity over disjoint document chunks. While computationally efficient, it exhibits critical failure modes when queries require multi-hop entity traversal, cross-document disambiguation, or indirect relationship inference.

This notebook evaluates **GraphRAG Engine**, a lightweight, local-first knowledge graph architecture that combines dense embeddings with **NetworkX Louvain community detection**. We evaluate both approaches across two quantized local LLMs (**Google Gemma-2-2B** and **Qwen-2.5-Coder-1.5B**) strictly constrained to a **6GB VRAM hardware ceiling**.

In [1]:
import json
import math
from collections import defaultdict

# Load ground-truth benchmark dataset
with open('../content/graphrag-benchmark.json', 'r') as f:
    benchmark = json.load(f)

meta = benchmark['benchmark_metadata']
print(f"[BENCHMARK INITIALIZED]: {meta['title']}")
print(f"Hardware Bound: {meta['hardware']}")
print(f"Evaluation Set: {meta['evaluation_set_size']} Ground-Truth Questions")
print(f"Modularity Metric: {meta['graph_clustering']}")

## 2. Experimental Architecture

```
                      [ Corpus: 30 Technical & Scientific Docs ]
                                      │
                ┌─────────────────────┴─────────────────────┐
                ▼                                           ▼
   [ Branch A: Plain Vector RAG ]             [ Branch B: Local GraphRAG ]
         Chunk Size: 512 tokens                     Entity Extraction via Spacy / LLM
         Overlap: 64 tokens                         Adjacency Graph via NetworkX
         Embedding: all-MiniLM-L6-v2                Community Partition: Louvain (Q=0.742)
         Top-K Retrieval: K=3                       Hierarchical Context Injection
                │                                           │
                └─────────────────────┬─────────────────────┘
                                      ▼
                    [ Local Ollama Inference (4-Bit Quantized) ]
                         • Gemma 2 (2B Q4_K_M: 1.8GB VRAM)
                         • Qwen 2.5 Coder (1.5B Q4_K_M: 1.3GB VRAM)
                                      ▼
                    [ Quantitative Evaluation Matrix ]
                         1. Answer Accuracy (0-100)
                         2. Hallucination Rate (%)
                         3. Entity Recall (%)
                         4. Latency & Time-To-First-Token (ms)
```

In [2]:
# Aggregate metrics display
metrics = benchmark['aggregate_metrics']
models = ['gemma2:2b', 'qwen2.5-coder:1.5b']

print("=" * 75)
print(f"{'Model':<20} | {'Method':<12} | {'Accuracy':<10} | {'Hallucination':<14} | {'Latency'}")
print("=" * 75)

for m in models:
    p = metrics[m]['plain_rag']
    g = metrics[m]['graph_rag']
    print(f"{m:<20} | {'Plain RAG':<12} | {p['accuracy']:<9.1f}% | {p['hallucination_rate']:<13.1f}% | {p['avg_latency_ms']}ms")
    print(f"{m:<20} | {'GraphRAG':<12} | {g['accuracy']:<9.1f}% | {g['hallucination_rate']:<13.1f}% | {g['avg_latency_ms']}ms")
    print(f"{'DELTA (' + m + ')':<20} | {'IMPROVEMENT':<12} | {metrics[m]['improvement']['accuracy_delta']:<10} | {metrics[m]['improvement']['hallucination_reduction']:<14} | +{g['avg_latency_ms']-p['avg_latency_ms']}ms overhead")
    print("-" * 75)

## 3. Key Findings & Quantitative Takeaways

1. **Hallucination Suppression (-75% to -85%):**  
   GraphRAG drastically reduces hallucinations from **22.9% down to 5.7%** on Gemma-2-2B, and from **20.0% down to 2.9%** on Qwen-2.5-Coder. In plain vector RAG, when relevant entities spanned across separated paragraphs or separate papers, the LLM hallucinated connections. In GraphRAG, the Louvain community summary supplied explicit graph path tokens.

2. **Multi-Hop Traversal Accuracy (+20% Absolute Gain):**  
   Overall answer accuracy rose from **68.6% to 88.6%** on Gemma, and from **71.4% to 91.4%** on Qwen.

3. **Compute Overhead within Consumer Bounds:**  
   Graph traversal and community summary lookups add **~210ms to ~220ms** latency overhead. On consumer RTX 3060 hardware, average total latency remained well under 1 second (840ms for Gemma, 690ms for Qwen), proving that GraphRAG is fully practical for local, privacy-first desktop workflows.

In [3]:
# Inspect sample qualitative evaluations
samples = benchmark['sample_evaluations']
for s in samples:
    print(f"\n[{s['id']}] {s['category'].upper()}")
    print(f"Question: {s['question']}")
    print(f"Plain RAG Result: {s['plain_rag']['response']}")
    print(f"Plain RAG Hallucination: {s['plain_rag']['hallucination']} ({s['plain_rag']['hallucination_note']})")
    print(f"GraphRAG Path: {' -> '.join(s['graph_rag']['traversed_nodes'])}")
    print(f"GraphRAG Result: {s['graph_rag']['response']}")
    print(f"Accuracy Score: {s['graph_rag']['accuracy_score']}/100")
    print("-" * 60)

## 4. Hardware Limitations & Honest Disclosure

- **VRAM Budget:** Quantized models require careful allocation. Running Gemma-2-2B at Q4_K_M occupies 1.8GB, leaving ~4.2GB VRAM. If large graph adjacency matrices exceeding 50,000 nodes are loaded into GPU tensor space, CUDA memory fragmentation occurs. Therefore, our implementation runs graph community partitioning exclusively on host CPU memory (16GB RAM) using NetworkX, streaming only relevant subgraphs into the context window.
- **Offline Guarantees:** 100% of embeddings, graph construction, and token generation occurred on the local machine with zero network requests.